# Лекция: Матрица корреляций в Python

**Дисциплина:** Введение в анализ больших данных

Темы:
- проверка нормальности (Shapiro–Wilk);
- ранговая корреляция Спирмена;
- матрица корреляций и heatmap;
- значимость коэффициентов;
- корреляции **по группам**.

Демо-данные — датасет **tips** (чаевые). Примеры **не совпадают** с лабораторным заданием: цель — освоить методы, задание выполните самостоятельно на своих данных.


## 0. Импорт и данные


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
sns.set_style("whitegrid")

tips = sns.load_dataset("tips")
print("Размерность:", tips.shape)
print(tips.head())


In [ ]:
quant_cols = ["total_bill", "tip", "size"]
print(tips[quant_cols].describe().round(2))


---
## 1. Проверка нормальности: Shapiro–Wilk

H0: выборка из нормального распределения.  
`stats.shapiro(x)` → (W, p-value).

При p > 0.05 обычно **не отвергают** H0.  
При больших n тест часто «ловит» даже мелкие отклонения — смотрите также гистограмму.


In [ ]:
print("=== Shapiro–Wilk ===")
for col in quant_cols:
    x = tips[col].dropna()
    W, p = stats.shapiro(x)
    verdict = "похоже на нормальное" if p > 0.05 else "отклонение от нормальности"
    print(f"{col:12s}: W={W:.4f}, p={p:.4g}  →  {verdict}")


Если переменные не нормальны, для связи часто берут **Спирмена** (ранги), а не Пирсона.


---
## 2. Корреляция Спирмена для пары переменных

`stats.spearmanr(x, y)` → (rho, p-value).


In [ ]:
rho, p = stats.spearmanr(tips["total_bill"], tips["tip"])
print(f"Spearman: total_bill ~ tip")
print(f"  rho = {rho:.4f}")
print(f"  p   = {p:.4g}")
print("  → значима" if p < 0.05 else "  → не значима")


---
## 3. Матрица корреляций и heatmap

`df.corr(method="spearman")` + `sns.heatmap`.


In [ ]:
corr = tips[quant_cols].corr(method="spearman")
print(corr.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(corr, annot=False, cmap="RdYlBu_r", center=0,
            vmin=-1, vmax=1, square=True, ax=axes[0])
axes[0].set_title("Спирмен (цвета)")

sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0,
            vmin=-1, vmax=1, square=True, ax=axes[1])
axes[1].set_title("Спирмен + числа")
plt.tight_layout()
plt.show()


---
## 4. Значимость всех пар

Для каждой пары столбцов можно получить p-value через `spearmanr`.


In [ ]:
def correlation_pvalues(df, method="spearman"):
    cols = list(df.columns)
    n = len(cols)
    pmat = pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)
    for i in range(n):
        for j in range(i + 1, n):
            if method == "spearman":
                _, p = stats.spearmanr(df.iloc[:, i], df.iloc[:, j], nan_policy="omit")
            else:
                common = df[[cols[i], cols[j]]].dropna()
                _, p = stats.pearsonr(common.iloc[:, 0], common.iloc[:, 1])
            pmat.iloc[i, j] = p
            pmat.iloc[j, i] = p
    return pmat

pvals = correlation_pvalues(tips[quant_cols])
print("p-value (Спирмен):")
print(pvals.round(4))


In [ ]:
print("Пары с |r| < 0.8 (пример порога):\n")
for i, c1 in enumerate(quant_cols):
    for c2 in quant_cols[i+1:]:
        r = corr.loc[c1, c2]
        p = pvals.loc[c1, c2]
        if abs(r) < 0.8:
            sig = "значима" if p < 0.05 else "не значима"
            print(f"  {c1} – {c2}: r={r:.3f}, p={p:.4g} → {sig}")


---
## 5. Корреляции **по группам** (например, по полу)


In [ ]:
for g in tips["sex"].dropna().unique():
    sub = tips.loc[tips["sex"] == g, quant_cols]
    c = sub.corr(method="spearman")
    print(f"\n=== sex = {g} (n={len(sub)}) ===")
    print(c.round(3))


In [ ]:
genders = tips["sex"].dropna().unique()
fig, axes = plt.subplots(1, len(genders), figsize=(5 * len(genders), 4))
if len(genders) == 1:
    axes = [axes]
for ax, g in zip(axes, genders):
    sub = tips.loc[tips["sex"] == g, quant_cols]
    sns.heatmap(sub.corr(method="spearman"), annot=True, fmt=".2f",
                cmap="RdYlBu_r", center=0, vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title(f"sex = {g}")
plt.tight_layout()
plt.show()


---
## 6. Связь одной переменной с остальными **по уровням фактора**

Пример: корреляция `tip` с `total_bill` и `size` отдельно для обеда и ужина.


In [ ]:
others = ["total_bill", "size"]
rows = []
for time in tips["time"].dropna().unique():
    sub = tips.loc[tips["time"] == time]
    for col in others:
        r, p = stats.spearmanr(sub["tip"], sub[col], nan_policy="omit")
        rows.append({"time": time, "variable": col, "rho": r, "p": p, "n": len(sub)})

res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))


In [ ]:
pivot = res.pivot(index="time", columns="variable", values="rho")
plt.figure(figsize=(6, 3))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlBu_r", center=0)
plt.title("Spearman: tip ~ ... по time")
plt.tight_layout()
plt.show()


---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Shapiro–Wilk | `stats.shapiro(x)` → (W, p) |
| Спирмен (пара) | `stats.spearmanr(x, y)` → (rho, p) |
| Матрица Спирмена | `df.corr(method="spearman")` |
| Heatmap | `sns.heatmap(corr, annot=True, cmap="RdYlBu_r", center=0)` |
| По группам | `df.groupby("g")[cols].corr(...)` или цикл |
| p-value пар | цикл / функция с `spearmanr` |

---
## Что сделать после лекции

1. Повторите Shapiro, матрицу и heatmap на **других** столбцах.
2. Откройте лабораторное задание и выполните его **самостоятельно** на указанном там наборе.
3. При большом n интерпретируйте не только p-value, но и **величину** коэффициента.

Удачи!
